# Phase 7 calibration results

Reads the artifacts produced by `python -m oath_score.calibration`:
* `data/processed/phase7_alpha_grid.csv` — α leave-one-out sweep across training cycles
* `data/processed/phase7_n_sensitivity.csv` — N sensitivity at α* on 2024
* `data/processed/phase7_cycle_ablation.csv` — 2014 ablation
* `data/processed/phase7_summary.json` — chosen α*, snapshot, feature set
* `data/processed/decile_thresholds.json` — frozen 1–10 cutpoints

**Headline finding:** dropping 2014 from the training set lifts the close-race metric by **+0.10pp** at T-60 (0.798 → 0.898). Pre-Trump-era patterns don't transfer to 2022/2024.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROC = Path('../data/processed')
alpha_grid = pd.read_csv(PROC / 'phase7_alpha_grid.csv')
n_sens = pd.read_csv(PROC / 'phase7_n_sensitivity.csv')
ablation = pd.read_csv(PROC / 'phase7_cycle_ablation.csv')
summary = json.loads((PROC / 'phase7_summary.json').read_text())
deciles = json.loads((PROC / 'decile_thresholds.json').read_text())
alpha_star = summary['alpha_star']
print(f'α* = {alpha_star}')
print(f'feature_set = {summary["feature_set"]}')
print(f'snapshot = {summary["snapshot"]}')
print(f'train cycles = {summary["train_cycles"]}')

## 1. α grid (LOO across training cycles)

Each row averages the close-race metric across leave-one-out folds. The α that maximizes this is α\*. The mean floor-saturation column shows that higher α directs more dollars to under-floor candidates — at the cost of close-race precision.

In [ ]:
alpha_grid.style.format({
    'mean_close_race': '{:.4f}',
    'std_close_race': '{:.4f}',
    'mean_pivotal': '{:.3f}',
    'mean_floor_sat': '{:.3f}',
}).set_caption(f'α grid LOO — best α* = {alpha_star}')

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(alpha_grid['alpha'], alpha_grid['mean_close_race'], '-o', color='tab:blue',
         label='Close-race metric (LOO mean)', linewidth=2)
ax1.fill_between(alpha_grid['alpha'],
                  alpha_grid['mean_close_race'] - alpha_grid['std_close_race'],
                  alpha_grid['mean_close_race'] + alpha_grid['std_close_race'],
                  alpha=0.15, color='tab:blue')
ax1.set_xlabel('α (financial-need weight)')
ax1.set_ylabel('Close-race metric', color='tab:blue')
ax1.set_xscale('symlog', linthresh=0.1)
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.axvline(alpha_star, color='black', linestyle='--', alpha=0.5, label=f'α* = {alpha_star}')

ax2 = ax1.twinx()
ax2.plot(alpha_grid['alpha'], alpha_grid['mean_floor_sat'], '-s', color='tab:orange',
         label='Floor-saturation (LOO mean)', linewidth=2)
ax2.set_ylabel('Floor-saturation efficiency', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')

ax1.set_title('α grid: close-race metric vs floor-saturation trade-off')
ax1.legend(loc='upper right')
ax2.legend(loc='center right')
ax1.grid(True, alpha=0.3)
plt.tight_layout()

## 2. N sensitivity at α\* (on held-out 2024)

How does the headline metric change with the top-N parameter? Includes the Cook-final benchmark for comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n_sens['n'], n_sens['model_score'], '-o', color='tab:blue', label='Model', linewidth=2.5)
ax.plot(n_sens['n'], n_sens['fundraising_score'], '--s', color='gray', label='Fundraising baseline', linewidth=1.5)
ax.plot(n_sens['n'], n_sens['cook_final_score'], '-.^', color='tab:red', label='Cook-final', linewidth=1.5)
ax.axhline(1.0, color='black', linestyle=':', alpha=0.4, label='Oracle')
ax.set_xscale('log')
ax.set_xlabel('Top-N')
ax.set_ylabel('% of $ to <5% races')
ax.set_title(f'N sensitivity at α* = {alpha_star} (2024 T-60, full feature set)')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
n_sens.style.format({
    'model_score': '{:.4f}', 'fundraising_score': '{:.4f}',
    'oracle_score': '{:.0f}', 'cook_final_score': '{:.4f}',
})

## 3. 2014-cycle ablation

Comparison: training with vs without 2014 at α\*. Pre-Trump-era patterns may not transfer; this is the empirical test.

In [ ]:
ablation.style.format({
    'headline_close_race': '{:.4f}',
    'headline_cook_final': '{:.4f}',
    'pivotal_dollar_share': '{:.3f}',
    'floor_saturation_efficiency': '{:.3f}',
}).set_caption('2014-cycle ablation at α*')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(ablation))
width = 0.35
ax.bar(x - width/2, ablation['headline_close_race'], width, label='Model close-race', color='tab:blue')
ax.bar(x + width/2, ablation['headline_cook_final'], width, label='Cook-final', color='tab:red')
ax.set_xticks(x)
ax.set_xticklabels(ablation['label'])
ax.set_ylabel('% of $ to <5% races')
ax.set_title('2014-cycle ablation: which training cycles help?')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()

## 4. Decile thresholds

The Streamlit UI displays a 1–10 score. Cutpoints are calibrated once on training cycles' impact_continuous distribution and **frozen** — they don't re-fit each cycle. A year with no toss-ups would have very few candidates score 10. Scores are comparable across cycles.

In [ ]:
cutpoints = deciles['cutpoints']
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 10), cutpoints, '-o', color='tab:blue', linewidth=2)
for i, c in enumerate(cutpoints, start=1):
    ax.annotate(f'{c:.3f}', (i, c), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax.set_xticks(range(1, 10))
ax.set_xticklabels([f'≤{i}' for i in range(1, 10)])
ax.set_xlabel('Decile boundary (cutpoint i separates bin i from bin i+1)')
ax.set_ylabel('Impact-continuous cutpoint')
ax.set_title(f'Frozen decile cutpoints (calibrated on cycles {deciles["cycles_calibrated_on"]})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
print('Cutpoints:', [f'{c:.4f}' for c in cutpoints])

## 5. Final calibrated config

* **α\*** = (see above)
* **Training cycles\*** = (2016, 2022) — 2014 dropped per ablation
* **Snapshot** = T-60 default for the headline number
* **Feature set** = `full`
* **Model** = multi-quantile
* **Universe** = all
* **N** = headline_n=10
* **Decile cutpoints** — frozen at `data/processed/decile_thresholds.json`

> ⚠️ **Numbers in this notebook are pre-audit.** The Cook-final benchmark column below uses the original raw-ordinal allocation, which had a structural bug that pulled Solid-D districts ahead of Toss-ups. The corrected, audit-honest headline lives in the [README "Headline numbers"](../README.md#headline-numbers) section.

### Headline numbers (audit-corrected — see README)

After fixing the Cook-final allocation to use symmetric Toss-up-distance weighting (`4 - |ord - 4|`) and integrating Wayback-Machine-cleaned 2016 ratings, model and Cook are **statistically indistinguishable** on close-race accuracy:

| Snapshot | Model [95% CI] | Cook-final [95% CI] | Paired Δ (model − Cook) [95% CI] |
|---|---:|---:|---:|
| T-110 | 0.911 [0.65, 1.00] | 0.900 [0.70, 1.00] | +0.040 [−0.245, +0.300] |
| T-60  | 0.907 [0.61, 1.00] | 0.900 [0.60, 1.00] | −0.002 [−0.335, +0.300] |
| T-20  | 0.910 [0.53, 1.00] | 0.900 [0.70, 1.00] | −0.007 [−0.382, +0.300] |

**Defensible claim:** the model uses inputs available 53 days before Cook's final ratings and matches Cook's accuracy on close-race allocation. Not "beats Cook." The pre-audit "+60pp" claim was a benchmark allocation bug, not a real lift — see README §[Validity checks](../README.md#validity-checks) for the audit narrative.